# RnD-5: LATE, 2SLS, CUPAC и off-policy сравнение политик

Этот ноутбук демонстрирует пять подходов для анализа пилота с **рандомизированным предложением** и **неполным принятием**:

1. **ITT / ATE по назначению** — эффект самого факта предложения;
2. **Wald LATE** — локальный эффект на принявших предложение;
3. **2SLS с ковариатами** — LATE с использованием признаков пользователей;
4. **CUPAC + Wald** — снижение дисперсии целевой переменной, затем Wald;
5. **CUPAC + 2SLS** — снижение дисперсии, затем 2SLS;
6. **Переосмысление задачи как замены политики назначения коммуникации** и сравнение `random` против `targeted` с помощью стандартных алгоритмов off-policy evaluation.

## Важные замечания

- В ноутбуке используется **синтетическая** схема данных, но она отражает ваш сценарий.
- Для простоты рассматривается **one-sided noncompliance**: пользователь может принять предложение только если получил коммуникацию.


## Теоретическая опора

- LATE и Wald-оценка для рандомизированного encouragement design;
- 2SLS с ковариатами как расширение IV-подхода;
- CUPED/CUPAC как методы снижения дисперсии;
- Off-policy evaluation (IPS / DM / DR) как способ сравнивать политики назначения на уже собранных логах.


In [2]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.base import clone

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
SEED = 42


## 1. Генератор синтетических данных

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def simulate_encouragement_pilot(
    n=30000,
    p_offer=0.5,
    late_effect=3.0,
    noise_std=2.5,
    seed=42,
):
    rng = np.random.default_rng(seed)

    x1 = rng.normal(size=n)
    x2 = rng.normal(size=n)
    x3 = rng.binomial(1, 0.35, size=n)
    x4 = rng.normal(size=n)
    x5 = rng.normal(size=n)
    X = np.column_stack([x1, x2, x3, x4, x5])

    Z = rng.binomial(1, p_offer, size=n)

    score = -4.2 + 1.0*x1 + 0.8*x2 + 1.2*x3 - 0.5*x4 + 0.6*x5
    p_accept = sigmoid(score)

    D = Z * rng.binomial(1, p_accept)

    mu = 1.5 + 0.8*x1 - 1.0*x2 + 0.6*x3 + 0.5*x4 - 0.3*x5 + 0.4*x1*x2
    Y = mu + late_effect * D + rng.normal(0, noise_std, size=n)

    df = pd.DataFrame({
        "x1": x1, "x2": x2, "x3": x3, "x4": x4, "x5": x5,
        "Z": Z, "D": D, "Y": Y, "p_accept": p_accept, "mu": mu
    })
    return df


In [4]:
df = simulate_encouragement_pilot()
df.head()


,x1,x2,x3,x4,x5,Z,D,Y,p_accept,mu
0,0.304717,-1.011428,0,0.005789,0.735584,1,0,-2.724364,0.013844,2.414141
1,-1.039984,-0.047397,0,1.080434,-0.525103,0,0,2.877601,0.002165,1.432875
2,0.750451,-1.346366,1,-0.833757,-0.320794,0,0,3.319999,0.043015,3.321934
3,0.940565,-0.715356,1,-0.253303,0.254859,0,0,7.588675,0.086895,3.095563
4,-1.951035,-0.244191,1,-0.076878,0.365674,1,0,-0.225574,0.007476,0.825793


In [5]:
pd.DataFrame({
    "metric": [
        "N",
        "share offered",
        "share uptake overall",
        "share uptake among treated",
        "mean outcome treated (Z=1)",
        "mean outcome control (Z=0)"
    ],
    "value": [
        len(df),
        df["Z"].mean(),
        df["D"].mean(),
        df.loc[df["Z"] == 1, "D"].mean(),
        df.loc[df["Z"] == 1, "Y"].mean(),
        df.loc[df["Z"] == 0, "Y"].mean(),
    ]
})


,metric,value
0,N,30000.000000
1,share offered,0.500133
2,share uptake overall,0.027233
3,share uptake among treated,0.054452
4,mean outcome treated (Z=1),1.903567
5,mean outcome control (Z=0),1.713587


## 2. Базовые оцениватели: ITT и Wald LATE

In [6]:
def itt_effect(y, z):
    y = np.asarray(y)
    z = np.asarray(z)
    return y[z == 1].mean() - y[z == 0].mean()

def wald_late(y, d, z):
    return itt_effect(y, z) / itt_effect(d, z)

itt_y = itt_effect(df["Y"], df["Z"])
itt_d = itt_effect(df["D"], df["Z"])
wald = wald_late(df["Y"], df["D"], df["Z"])

pd.DataFrame({
    "estimator": ["ITT on outcome", "ITT on take-up", "Wald LATE"],
    "value": [itt_y, itt_d, wald]
})


,estimator,value
0,ITT on outcome,0.189980
1,ITT on take-up,0.054452
2,Wald LATE,3.488933


## 3. 2SLS с ковариатами

In [7]:
def add_const(X):
    X = np.asarray(X)
    return np.column_stack([np.ones(len(X)), X])

def ols_fit(X, y):
    beta = np.linalg.pinv(X.T @ X) @ (X.T @ y)
    yhat = X @ beta
    resid = y - yhat
    n, p = X.shape
    sigma2 = resid.T @ resid / max(n - p, 1)
    vcov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(vcov))
    return beta, se, yhat

def tsls_single_iv(y, d, z, X):
    X = np.asarray(X)
    y = np.asarray(y)
    d = np.asarray(d)
    z = np.asarray(z)

    W1 = add_const(np.column_stack([z, X]))
    beta1, se1, d_hat = ols_fit(W1, d)

    W2 = add_const(np.column_stack([d_hat, X]))
    beta2, se2, y_hat = ols_fit(W2, y)

    return {
        "coef": beta2[1],
        "se_naive": se2[1],
        "first_stage_coef_z": beta1[1],
        "first_stage_se_z": se1[1]
    }

X = df[["x1", "x2", "x3", "x4", "x5"]].values
iv_res = tsls_single_iv(df["Y"].values, df["D"].values, df["Z"].values, X)

pd.DataFrame({
    "metric": ["2SLS coef on D_hat", "naive SE", "first-stage coef on Z", "first-stage naive SE"],
    "value": [iv_res["coef"], iv_res["se_naive"], iv_res["first_stage_coef_z"], iv_res["first_stage_se_z"]]
})


,metric,value
0,2SLS coef on D_hat,3.174745
1,naive SE,0.541011
2,first-stage coef on Z,0.054902
3,first-stage naive SE,0.001808


## 4. CUPAC: снижение дисперсии целевой переменной

In [8]:
def cross_fitted_prediction(X, y, model, n_splits=5, seed=42):
    X = np.asarray(X)
    y = np.asarray(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pred = np.zeros(len(y))
    for train_idx, test_idx in kf.split(X):
        m = clone(model)
        m.fit(X[train_idx], y[train_idx])
        pred[test_idx] = m.predict(X[test_idx])
    return pred

cupac_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=50,
    random_state=SEED,
    n_jobs=-1
)

y_hat_cupac = cross_fitted_prediction(X, df["Y"].values, model=cupac_model, n_splits=5, seed=SEED)
df["Y_cupac"] = df["Y"] - y_hat_cupac

pd.DataFrame({
    "metric": ["var(Y)", "var(Y_cupac)"],
    "value": [df["Y"].var(), df["Y_cupac"].var()]
})


,metric,value
0,var(Y),8.697097
1,var(Y_cupac),6.536824


## 5. Wald после CUPAC

In [9]:
wald_cupac = wald_late(df["Y_cupac"], df["D"], df["Z"])
pd.DataFrame({
    "estimator": ["Wald LATE on raw Y", "Wald LATE on CUPAC residualized Y"],
    "value": [wald, wald_cupac]
})


,estimator,value
0,Wald LATE on raw Y,3.488933
1,Wald LATE on CUPAC residualized Y,3.177347


## 6. 2SLS после CUPAC

In [10]:
iv_cupac_res = tsls_single_iv(df["Y_cupac"].values, df["D"].values, df["Z"].values, X)
pd.DataFrame({
    "metric": ["2SLS coef on D_hat (raw Y)", "2SLS coef on D_hat (CUPAC Y)", "naive SE raw", "naive SE CUPAC"],
    "value": [iv_res["coef"], iv_cupac_res["coef"], iv_res["se_naive"], iv_cupac_res["se_naive"]]
})


,metric,value
0,2SLS coef on D_hat (raw Y),3.174745
1,2SLS coef on D_hat (CUPAC Y),3.120033
2,naive SE raw,0.541011
3,naive SE CUPAC,0.537029


## 7. Переосмысление задачи как сравнения политик назначения

Теперь задача формулируется не как оценка LATE, а как сравнение двух политик:
- **random**: случайная отправка коммуникации с той же интенсивностью, что и в пилоте;
- **targeted**: отправка коммуникации пользователям с высокой предсказанной склонностью к акцепту.


In [11]:
df["action"] = df["Z"]
df["reward"] = df["Y"]

treated_mask = df["Z"] == 1
accept_model = LogisticRegression(max_iter=1000)
accept_model.fit(df.loc[treated_mask, ["x1", "x2", "x3", "x4", "x5"]], df.loc[treated_mask, "D"])

df["accept_score"] = accept_model.predict_proba(df[["x1", "x2", "x3", "x4", "x5"]])[:, 1]

logging_prob = df["Z"].mean()
threshold = np.quantile(df["accept_score"], 1 - logging_prob)
df["pi_targeted"] = (df["accept_score"] >= threshold).astype(float)
df["pi_random"] = logging_prob

pd.DataFrame({
    "metric": ["logging_prob", "targeted offer rate"],
    "value": [logging_prob, df["pi_targeted"].mean()]
})


,metric,value
0,logging_prob,0.500133
1,targeted offer rate,0.500133


## 8. Off-policy evaluation: IPS, DM, DR

In [12]:
def ips_estimate(reward, action, pscore, pi_e):
    reward = np.asarray(reward)
    action = np.asarray(action)
    pscore = np.asarray(pscore)
    pi_e = np.asarray(pi_e)
    w = np.where(action == 1, pi_e / pscore, (1 - pi_e) / (1 - pscore))
    return np.mean(w * reward)

def fit_reward_models(df, features):
    model0 = RandomForestRegressor(n_estimators=200, min_samples_leaf=50, random_state=1, n_jobs=-1)
    model1 = RandomForestRegressor(n_estimators=200, min_samples_leaf=50, random_state=2, n_jobs=-1)

    X0 = df.loc[df["action"] == 0, features].values
    y0 = df.loc[df["action"] == 0, "reward"].values
    X1 = df.loc[df["action"] == 1, features].values
    y1 = df.loc[df["action"] == 1, "reward"].values

    model0.fit(X0, y0)
    model1.fit(X1, y1)
    return model0, model1

def dm_estimate(df, features, pi_e):
    model0, model1 = fit_reward_models(df, features)
    X = df[features].values
    mu0 = model0.predict(X)
    mu1 = model1.predict(X)
    return np.mean(pi_e * mu1 + (1 - pi_e) * mu0)

def dr_estimate(df, features, pi_e, pscore):
    model0, model1 = fit_reward_models(df, features)
    X = df[features].values
    action = df["action"].values
    reward = df["reward"].values
    mu0 = model0.predict(X)
    mu1 = model1.predict(X)
    mu_a = np.where(action == 1, mu1, mu0)
    w = np.where(action == 1, pi_e / pscore, (1 - pi_e) / (1 - pscore))
    return np.mean(pi_e * mu1 + (1 - pi_e) * mu0 + w * (reward - mu_a))

features = ["x1", "x2", "x3", "x4", "x5"]
pscore = np.repeat(logging_prob, len(df))

ope_res = pd.DataFrame({
    "policy": ["random", "targeted", "random", "targeted", "random", "targeted"],
    "estimator": ["IPS", "IPS", "DM", "DM", "DR", "DR"],
    "value": [
        ips_estimate(df["reward"], df["action"], pscore, np.repeat(df["pi_random"].iloc[0], len(df))),
        ips_estimate(df["reward"], df["action"], pscore, df["pi_targeted"].values),
        dm_estimate(df, features, np.repeat(df["pi_random"].iloc[0], len(df))),
        dm_estimate(df, features, df["pi_targeted"].values),
        dr_estimate(df, features, np.repeat(df["pi_random"].iloc[0], len(df)), pscore),
        dr_estimate(df, features, df["pi_targeted"].values, pscore),
    ]
})
ope_res


,policy,estimator,value
0,random,IPS,1.808602
1,targeted,IPS,1.868665
2,random,DM,1.805228
3,targeted,DM,1.877631
4,random,DR,1.806980
5,targeted,DR,1.872096


In [13]:
pivot_ope = ope_res.pivot(index="estimator", columns="policy", values="value")
pivot_ope["targeted_minus_random"] = pivot_ope["targeted"] - pivot_ope["random"]
pivot_ope


policy,random,targeted,targeted_minus_random
estimator,,,
DM,1.805228,1.877631,0.072403
DR,1.806980,1.872096,0.065116
IPS,1.808602,1.868665,0.060063


## 9. Сводная таблица

In [14]:
summary = pd.DataFrame({
    "approach": [
        "ITT / ATE by assignment",
        "Wald LATE",
        "2SLS with covariates",
        "CUPAC + Wald",
        "CUPAC + 2SLS",
        "Off-policy random vs targeted"
    ],
    "what_it_estimates": [
        "эффект назначения коммуникации",
        "LATE на compliers",
        "LATE с учетом ковариат",
        "LATE после снижения дисперсии outcome",
        "LATE с ковариатами и снижением дисперсии",
        "ценность замены политики назначения"
    ],
    "main_number": [
        itt_y,
        wald,
        iv_res["coef"],
        wald_cupac,
        iv_cupac_res["coef"],
        pivot_ope.loc["DR", "targeted_minus_random"]
    ]
})
summary


,approach,what_it_estimates,main_number
0,ITT / ATE by assignment,эффект назначения коммуникации,0.189980
1,Wald LATE,LATE на compliers,3.488933
2,2SLS with covariates,LATE с учетом ковариат,3.174745
3,CUPAC + Wald,LATE после снижения дисперсии outcome,3.177347
4,CUPAC + 2SLS,LATE с ковариатами и снижением дисперсии,3.120033
5,Off-policy random vs targeted,ценность замены политики назначения,0.065116


## 10. Основные выводы

In [15]:
print("1. ITT по назначению действительно размывает эффект, когда акцепт среди получивших коммуникацию низкий.")
print("2. Wald LATE восстанавливает эффект на тех, кто реально принимает предложение благодаря коммуникации.")
print("3. 2SLS позволяет добавить ковариаты и обычно даёт более устойчивую оценку, чем голый Wald.")
print("4. CUPAC можно использовать как способ уменьшить дисперсию outcome перед IV-оцениванием.")
print("5. Off-policy блок отвечает на отдельный вопрос: стоит ли менять политику назначения коммуникации с random на targeted.")


1. ITT по назначению действительно размывает эффект, когда акцепт среди получивших коммуникацию низкий.
2. Wald LATE восстанавливает эффект на тех, кто реально принимает предложение благодаря коммуникации.
3. 2SLS позволяет добавить ковариаты и обычно даёт более устойчивую оценку, чем голый Wald.
4. CUPAC можно использовать как способ уменьшить дисперсию outcome перед IV-оцениванием.
5. Off-policy блок отвечает на отдельный вопрос: стоит ли менять политику назначения коммуникации с random на targeted.
